# EWS Feature Engineering

This notebook creates a ML feature table for the Early Warning System.

The previous preprocessing notebook produced a clean baseline modeling dataset. This notebook enriches it with business-relevant and leakage-safe features from product, inventory tables.

Main steps:

1. Load the cleaned product-month dataset and raw supporting tables.
2. Join product metadata, inventory details.
3. Create ML features:
   - product age and lifecycle features
   - margin and gross-profit features
   - campaign spend and digital demand features
   - lead-time and stockout pressure features
   - seasonality and macro-market features
   - category-region benchmark features
   - relative product performance features
   - rolling momentum, acceleration, and stability features
4. Avoid target leakage:
   - no future target columns are used as features
   - rolling benchmark features use historical or current-month observable data only
   - the final month with missing future label is kept for scoring but excluded from train/validation/test
5. Export:
   - a human-readable feature-engineered dataset
   - train/validation/test ML-ready matrices
   - ID files for traceability
   - a compact feature engineering quality report

## Step Overview

Before the code runs, here is the logic in plain language:

- We start from the cleaned baseline table because it already contains safe lag, rolling, and target fields.
- We enrich it with product metadata, inventory lead times, campaign spend, product page views, seasonality, and macro-healthcare context.
- We create stronger features that explain *why* a product may be risky, not only whether sales dropped.
- We fit imputation, scaling, and one-hot encoding only on the training period, then apply the same transformations to validation and test.
- We keep IDs separately so predictions can be traced back to product, month, and region.


In [3]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RANDOM_SEED = 42

INPUT_DIR = Path("../data/preprocessed")
OUTPUT_DIR = Path("../data/feature_engineering")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "next_month_risk_label"
ID_COLS = ["product_id", "year_month", "region"]

In [5]:
# Divide safely and return NaN where denominator is zero or missing
def safe_divide(numerator, denominator):
    return numerator / denominator.replace(0, np.nan)


# Linear trend slope for a rolling window
def slope(values):
    values = np.asarray(values, dtype=float)
    if len(values) < 2 or np.all(np.isnan(values)):
        return np.nan
    x = np.arange(len(values))
    return float(np.polyfit(x, values, 1)[0])